# Debug Data Notebook

Inspect and validate all four prepared datasets:
- BeaverTails (training data)
- XSTest (safe + unsafe splits)
- WildGuard (refusal evaluation)
- Protected Mix D_gen (subspace extraction)

Run after `scripts/01_prepare_data.sh`.

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd

# Add project root to path
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f]

print('Project root:', ROOT)

## 1. BeaverTails

In [ ]:
bt_train = load_jsonl(ROOT / 'artifacts/datasets/beavertails/train.jsonl')
bt_val = load_jsonl(ROOT / 'artifacts/datasets/beavertails/val.jsonl')

print(f'Train: {len(bt_train)} samples')
print(f'Val:   {len(bt_val)} samples')

# Label distribution
train_safe = sum(s['label'] for s in bt_train)
print(f'Train label distribution: safe={train_safe}, unsafe={len(bt_train)-train_safe}')

In [ ]:
# Inspect a few samples
print('--- Safe sample ---')
safe = next(s for s in bt_train if s['label'] == 1)
print('Prompt:', safe['prompt'][:100])
print('Response:', safe['response'][:200])

print()
print('--- Unsafe sample (canonical refusal response) ---')
unsafe = next(s for s in bt_train if s['label'] == 0)
print('Prompt:', unsafe['prompt'][:100])
print('Response:', unsafe['response'][:200])

In [ ]:
# Verify chat template format
sample = bt_train[0]
print('Messages format:')
for m in sample.get('messages', []):
    print(f'  [{m["role"]}]: {m["content"][:80]}...')

## 2. XSTest

In [ ]:
xs_safe = load_jsonl(ROOT / 'artifacts/datasets/xstest/safe.jsonl')
xs_unsafe = load_jsonl(ROOT / 'artifacts/datasets/xstest/unsafe.jsonl')

print(f'Safe prompts: {len(xs_safe)}')
print(f'Unsafe prompts: {len(xs_unsafe)}')

# Type distribution
safe_types = pd.Series([s.get('type', 'unknown') for s in xs_safe]).value_counts()
print('\nSafe type distribution:')
print(safe_types)

In [ ]:
# Inspect examples
print('Safe prompt example:', xs_safe[0]['prompt'])
print('Unsafe prompt example:', xs_unsafe[0]['prompt'])

## 3. WildGuard

In [ ]:
wg_test = load_jsonl(ROOT / 'artifacts/datasets/wildguard/test.jsonl')

print(f'WildGuardTest: {len(wg_test)} samples')

df_wg = pd.DataFrame(wg_test)
print('\nLabel distributions:')
print(df_wg['prompt_harm_label'].value_counts())
print()
print(df_wg['response_refusal_label'].value_counts())

## 4. Protected Mix (D_gen)

In [ ]:
pm = load_jsonl(ROOT / 'artifacts/datasets/protected_mix/protected_mix.jsonl')

print(f'Protected mix: {len(pm)} samples')

sources = pd.Series([s.get('source', 'unknown') for s in pm]).value_counts()
print('\nSource distribution:')
print(sources)

In [ ]:
# Inspect format diversity
for source in ['mmlu_pro', 'gsm8k', 'hellaswag']:
    sample = next((s for s in pm if s.get('source') == source), None)
    if sample:
        print(f'--- {source} ---')
        print('Format:', sample['format'])
        print('Text:', sample['text'][:120])
        print()

## 5. Token Length Analysis

In [ ]:
# Approximate token length using whitespace tokenization
import numpy as np
import matplotlib.pyplot as plt

def token_len_approx(text):
    return len(text.split())

datasets = {
    'BeaverTails train': [token_len_approx(s['prompt'] + ' ' + s['response']) for s in bt_train],
    'Protected Mix': [token_len_approx(s['text']) for s in pm],
}

fig, ax = plt.subplots(figsize=(10, 4))
for name, lengths in datasets.items():
    arr = np.array(lengths)
    print(f'{name}: median={np.median(arr):.0f}, p95={np.percentile(arr,95):.0f}, max={arr.max()}')
    ax.hist(arr, bins=50, alpha=0.5, label=name, density=True)

ax.set_xlabel('Approx token length')
ax.set_ylabel('Density')
ax.set_title('Token length distribution')
ax.legend()
plt.tight_layout()
plt.show()